# Programma 2 — Estrazione di informazioni

Estrazione di informazioni dal **Corpus 2** (`Corpora_2.txt`): articoli scientifici, tre articoli di psicologia.

In [1]:
# Import delle librerie e download delle risorse NLTK

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk import FreqDist, bigrams, ngrams, ne_chunk
import math

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\user\AppData\Roaming\

True

## Lettura del corpus

In [2]:
# Lettura del corpus da analizzare
# Per analizzare l'altro corpus è sufficiente cambiare il nome del file e rieseguire

nome_file = "Corpora_2.txt"

with open(nome_file, "r", encoding="utf-8") as f:
    testo = f.read()

# Verifica: stampa dei primi 200 caratteri
print(f"=== Inizio di {nome_file} ===")
print(testo[:200])

=== Inizio di Corpora_2.txt ===
Introduction Social inequality and injustice are based on group memberships, like gender, race/ethnicity, and class. For example, women and gender minorities experience workplace discrimination, and r


## Preprocessing

In [3]:
# Suddivisione del testo in frasi
frasi = sent_tokenize(testo)

# Tokenizzazione: lista piatta di tutti i token del corpus
tokens_con_punteggiatura = [token for frase in frasi for token in word_tokenize(frase)]

# PoS tagging dell'intero corpus
pos_con_punteggiatura = pos_tag(tokens_con_punteggiatura)

# Rimozione della punteggiatura: si tengono solo i token che contengono
# almeno un carattere alfanumerico. In questo modo vengono esclusi sia i segni
# di punteggiatura standard sia quelli tipografici
tokens_pos = [(token, tag) for token, tag in pos_con_punteggiatura if any(c.isalnum() for c in token)]

# Token nella forma originale (con maiuscole), usati per le Named Entity
tokens = [token for token, tag in tokens_pos]

# Token normalizzati in minuscolo, usati per i conteggi lessicali
tokens_lower = [token.lower() for token in tokens]

print(f"Numero di frasi: {len(frasi)}")
print(f"Numero di token (senza punteggiatura): {len(tokens)}")

Numero di frasi: 825
Numero di token (senza punteggiatura): 20307


## 1. Top-50 sostantivi, verbi e aggettivi più frequenti

In [4]:
# Funzione che estrae i top-N token appartenenti a una categoria grammaticale,
# ordinati per frequenza decrescente. I tag PoS da considerare sono passati come lista
# I token vengono normalizzati in minuscolo per non distinguere le maiuscole

def top_per_categoria(tokens_pos, tag_validi, n=50):
    parole = [token.lower() for token, tag in tokens_pos if tag in tag_validi]
    return FreqDist(parole).most_common(n)

# Per i sostantivi si considerano tutti i tag nominali (nomi comuni e propri)
tag_sostantivi = ["NN", "NNS", "NNP", "NNPS"]
tag_verbi = ["VB", "VBD", "VBG", "VBN", "VBP", "VBZ"]
tag_aggettivi = ["JJ", "JJR", "JJS"]

top50_sostantivi = top_per_categoria(tokens_pos, tag_sostantivi)
top50_verbi = top_per_categoria(tokens_pos, tag_verbi)
top50_aggettivi = top_per_categoria(tokens_pos, tag_aggettivi)

print("=== Top-50 Sostantivi ===")
for parola, freq in top50_sostantivi:
    print(f"{parola}: {freq}")

print("\n=== Top-50 Verbi ===")
for parola, freq in top50_verbi:
    print(f"{parola}: {freq}")

print("\n=== Top-50 Aggettivi ===")
for parola, freq in top50_aggettivi:
    print(f"{parola}: {freq}")

=== Top-50 Sostantivi ===
gender: 127
participants: 127
individuals: 102
judgments: 100
black: 99
effect: 82
distress: 77
men: 76
group: 74
study: 72
treatment: 72
inequality: 71
status: 70
education: 69
income: 69
women: 66
attitudes: 66
class: 65
white: 63
people: 60
stereotypes: 59
response: 59
n400: 59
groups: 54
health: 53
biases: 53
results: 52
targets: 45
effects: 45
research: 44
target: 43
race: 43
mhps: 41
p: 40
countries: 38
sd: 36
dimensions: 35
data: 35
information: 32
studies: 30
evidence: 29
sample: 29
context: 28
norms: 28
prejudice: 28
analysis: 27
bias: 26
se: 26
americans: 25
psychopathology: 25

=== Top-50 Verbi ===
were: 132
was: 128
are: 91
is: 83
be: 80
have: 42
compared: 34
based: 27
had: 27
associated: 26
s: 26
using: 25
used: 23
showed: 23
found: 22
been: 20
did: 20
presented: 20
has: 18
recorded: 18
expected: 16
tested: 16
provide: 16
revealed: 16
test: 15
related: 15
believe: 15
including: 14
according: 14
influenced: 14
depending: 13
see: 13
included: 13
pro

## 2. Top-10 n-grammi più frequenti

In [5]:
# Funzione che estrae i top-10 n-grammi più frequenti per un dato n
# Si lavora sui token normalizzati in minuscolo (tokens_lower)

def top_ngrammi(tokens, n, top=10):
    n_grammi = list(ngrams(tokens, n))
    return FreqDist(n_grammi).most_common(top)

# Calcolo per n = 1, 2, 3
for n in [1, 2, 3]:
    print(f"=== Top-10 {n}-grammi ===")
    for ngramma, freq in top_ngrammi(tokens_lower, n):
        print(f"{ngramma}: {freq}")
    print()

=== Top-10 1-grammi ===
('the',): 931
('and',): 699
('of',): 575
('to',): 498
('in',): 431
('a',): 282
('that',): 221
('for',): 210
('on',): 186
('with',): 169

=== Top-10 2-grammi ===
('of', 'the'): 101
('in', 'the'): 96
('on', 'the'): 50
('to', 'the'): 49
('mental', 'health'): 45
('biases', 'in'): 39
('attitudes', 'towards'): 38
('the', 'n400'): 37
('by', 'the'): 33
('racial', 'biases'): 32

=== Top-10 3-grammi ===
('racial', 'biases', 'in'): 30
('attitudes', 'towards', 'men'): 23
('in', 'response', 'to'): 23
('than', 'white', 'individuals'): 18
('distress', 'and', 'treatment'): 17
('the', 'n400', 'response'): 17
('gender', 'and', 'class'): 16
('on', 'attitudes', 'towards'): 16
('black', 'and', 'white'): 15
('less', 'than', 'white'): 14



## 3. Top-30 n-grammi di PoS più frequenti

In [6]:
# Funzione che estrae i top-30 n-grammi di PoS più frequenti per un dato n
# Si considerano solo le sequenze di categorie grammaticali (i tag), non i token

def top_ngrammi_pos(tokens_pos, n, top=30):
    soli_tag = [tag for token, tag in tokens_pos]
    n_grammi = list(ngrams(soli_tag, n))
    return FreqDist(n_grammi).most_common(top)

# Calcolo per n = 1, 2, 3, 4, 5
for n in [1, 2, 3, 4, 5]:
    print(f"=== Top-30 {n}-grammi di PoS ===")
    for ngramma, freq in top_ngrammi_pos(tokens_pos, n):
        print(f"{ngramma}: {freq}")
    print()

=== Top-30 1-grammi di PoS ===
('NN',): 3709
('IN',): 2577
('JJ',): 2367
('NNS',): 2250
('DT',): 1572
('NNP',): 1289
('CC',): 807
('VBD',): 706
('CD',): 657
('RB',): 656
('VBN',): 642
('VB',): 519
('TO',): 498
('VBG',): 432
('VBP',): 289
('VBZ',): 241
('MD',): 199
('PRP',): 197
('JJR',): 134
('RBR',): 124
('WDT',): 104
('WRB',): 96
('PRP$',): 94
('NNPS',): 40
('EX',): 28
('FW',): 23
('WP',): 18
('JJS',): 14
('RP',): 5
('RBS',): 5

=== Top-30 2-grammi di PoS ===
('JJ', 'NN'): 1098
('NN', 'IN'): 868
('IN', 'DT'): 722
('NNS', 'IN'): 645
('JJ', 'NNS'): 645
('DT', 'NN'): 599
('DT', 'JJ'): 519
('NN', 'NN'): 510
('IN', 'NN'): 494
('IN', 'JJ'): 469
('NN', 'NNS'): 466
('NN', 'CC'): 343
('VBN', 'IN'): 318
('IN', 'NNS'): 287
('NNP', 'NNP'): 264
('NNP', 'CD'): 239
('TO', 'VB'): 229
('NN', 'VBD'): 227
('NNP', 'NN'): 221
('JJ', 'JJ'): 219
('CC', 'NN'): 209
('IN', 'NNP'): 205
('NN', 'NNP'): 201
('NNS', 'VBD'): 190
('CC', 'JJ'): 180
('NNS', 'VBP'): 164
('NNS', 'CC'): 163
('MD', 'VB'): 159
('VBD', 'VBN

## 4. Top-10 bigrammi Aggettivo + Sostantivo

In [7]:
# Estrazione dei bigrammi Aggettivo + Sostantivo dal corpus
# Si scorrono i bigrammi di token consecutivi e si tengono solo quelli in cui
# il primo token è un aggettivo (JJ, JJR, JJS) e il secondo un sostantivo (NN, NNS, NNP, NNPS)

tag_agg = ["JJ", "JJR", "JJS"]
tag_sost = ["NN", "NNS", "NNP", "NNPS"]

bigrammi_pos = list(bigrams(tokens_pos))

bigrammi_agg_sost = []
for (token1, tag1), (token2, tag2) in bigrammi_pos:
    if tag1 in tag_agg and tag2 in tag_sost:
        # il bigramma viene salvato come coppia di parole in minuscolo
        bigrammi_agg_sost.append((token1.lower(), token2.lower()))

# Frequenza dei bigrammi Agg+Sost
freq_bigrammi = FreqDist(bigrammi_agg_sost)

# 4.1 - Top-10 per frequenza decrescente
print("=== Top-10 bigrammi Agg+Sost per frequenza ===")
for bigramma, freq in freq_bigrammi.most_common(10):
    print(f"{bigramma}: {freq}")

=== Top-10 bigrammi Agg+Sost per frequenza ===
('mental', 'health'): 43
('racial', 'biases'): 32
('psychological', 'distress'): 28
('mental', 'healthcare'): 15
('distress', 'judgments'): 14
('low', 'income'): 12
('social', 'class'): 11
('high', 'education'): 11
('general', 'inequality'): 10
('frontal', 'positivity'): 10


In [8]:
# Per calcolare le probabilità servono le frequenze dei singoli token (in minuscolo)
# e il numero totale di token del corpus
freq_token = FreqDist(tokens_lower)
N = len(tokens_lower)

# 4.2 - Probabilità condizionata: P(sostantivo | aggettivo) = freq(agg, sost) / freq(agg)
prob_condizionata = {}
for bigramma, freq in freq_bigrammi.items():
    aggettivo = bigramma[0]
    prob_condizionata[bigramma] = freq / freq_token[aggettivo]

# 4.3 - Probabilità congiunta: P(aggettivo, sostantivo) = freq(agg, sost) / N
prob_congiunta = {}
for bigramma, freq in freq_bigrammi.items():
    prob_congiunta[bigramma] = freq / N

# Ordinamento e stampa dei top-10 per probabilità condizionata
print("=== Top-10 bigrammi Agg+Sost per probabilità condizionata ===")
for bigramma, prob in sorted(prob_condizionata.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{bigramma}: {prob:.4f}")

# Ordinamento e stampa dei top-10 per probabilità congiunta
print("\n=== Top-10 bigrammi Agg+Sost per probabilità congiunta ===")
for bigramma, prob in sorted(prob_congiunta.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{bigramma}: {prob:.6f}")

=== Top-10 bigrammi Agg+Sost per probabilità condizionata ===
('intimate', 'relationships'): 1.0000
('racial/ethnic', 'minorities'): 1.0000
('monolithic', 'approach'): 1.0000
('empirical', 'studies'): 1.0000
('productive', 'assets'): 1.0000
('surplus', 'labor'): 1.0000
('inextricable', 'aspect'): 1.0000
('ample', 'evidence'): 1.0000
('ambivalent', 'warmth'): 1.0000
('prestigious', 'occupation'): 1.0000

=== Top-10 bigrammi Agg+Sost per probabilità congiunta ===
('mental', 'health'): 0.002117
('racial', 'biases'): 0.001576
('psychological', 'distress'): 0.001379
('mental', 'healthcare'): 0.000739
('distress', 'judgments'): 0.000689
('low', 'income'): 0.000591
('social', 'class'): 0.000542
('high', 'education'): 0.000542
('general', 'inequality'): 0.000492
('frontal', 'positivity'): 0.000492


In [9]:
# 4.4 - MI (Mutual Information) e LMI (Local Mutual Information)
# MI(A,B)  = log2( freq(A,B) * N / (freq(A) * freq(B)) )
# LMI(A,B) = freq(A,B) * MI(A,B)

mi = {}
lmi = {}
for bigramma, freq in freq_bigrammi.items():
    aggettivo, sostantivo = bigramma
    # MI: rapporto tra co-occorrenza osservata e attesa, in scala logaritmica (base 2)
    valore_mi = math.log2((freq * N) / (freq_token[aggettivo] * freq_token[sostantivo]))
    mi[bigramma] = valore_mi
    # LMI: MI pesata per la frequenza del bigramma
    lmi[bigramma] = freq * valore_mi

# Top-10 per MI
print("=== Top-10 bigrammi Agg+Sost per MI ===")
top_mi = sorted(mi.items(), key=lambda x: x[1], reverse=True)[:10]
for bigramma, valore in top_mi:
    print(f"{bigramma}: {valore:.4f}")

# Top-10 per LMI
print("\n=== Top-10 bigrammi Agg+Sost per LMI ===")
top_lmi = sorted(lmi.items(), key=lambda x: x[1], reverse=True)[:10]
for bigramma, valore in top_lmi:
    print(f"{bigramma}: {valore:.4f}")

# Numero di elementi comuni tra le due liste (top-10 MI e top-10 LMI)
bigrammi_mi = set(bigramma for bigramma, valore in top_mi)
bigrammi_lmi = set(bigramma for bigramma, valore in top_lmi)
comuni = bigrammi_mi & bigrammi_lmi
print(f"\nNumero di elementi comuni tra le due liste (MI e LMI): {len(comuni)}")

=== Top-10 bigrammi Agg+Sost per MI ===
('productive', 'assets'): 14.3097
('inextricable', 'aspect'): 14.3097
('ambivalent', 'warmth'): 14.3097
('communal', 'feminine'): 14.3097
('anonymized', 'datasets'): 14.3097
('external', 'validity'): 14.3097
('functional', 'equivalence'): 14.3097
('ethical', 'guidelines'): 14.3097
('spearman-brown', 'consistency'): 14.3097
('labour', 'force'): 14.3097

=== Top-10 bigrammi Agg+Sost per LMI ===
('mental', 'health'): 328.2188
('racial', 'biases'): 241.1960
('psychological', 'distress'): 211.7622
('mental', 'healthcare'): 113.9042
('frontal', 'positivity'): 96.5434
('low', 'income'): 78.4584
('neutral', 'ones'): 75.9673
('current', 'work'): 75.5146
('general', 'inequality'): 70.2244
('distress', 'judgments'): 68.0249

Numero di elementi comuni tra le due liste (MI e LMI): 0


## 5. Analisi delle frasi (lunghezza 5-15 token, almeno metà non hapax)

In [10]:
# Frequenza di ogni token nel corpus (in minuscolo), usata per distinguere hapax e non hapax

freq_corpus = FreqDist(tokens_lower)

# Per ogni frase del corpus si ricavano i suoi token (in minuscolo, senza punteggiatura)
# Si selezionano poi le frasi che rispettano due condizioni:
#   - lunghezza compresa tra 5 e 15 token
#   - almeno la metà dei token NON è hapax 

frasi_idonee = []
for frase in frasi:
    token_frase = [t.lower() for t in word_tokenize(frase) if any(c.isalnum() for c in t)]

    if 5 <= len(token_frase) <= 15:
        non_hapax = [t for t in token_frase if freq_corpus[t] >= 2]
        if len(non_hapax) >= len(token_frase) / 2:
            frasi_idonee.append(token_frase)

print(f"Numero di frasi idonee: {len(frasi_idonee)}")
print("\nEsempio di frasi idonee:")
for frase in frasi_idonee[:3]:
    print(frase)

Numero di frasi idonee: 203

Esempio di frasi idonee:
['introduction', 'social', 'inequality', 'and', 'injustice', 'are', 'based', 'on', 'group', 'memberships', 'like', 'gender', 'race/ethnicity', 'and', 'class']
['similarly', 'racism', 'harms', 'people', 's', 'health', 'and', 'healthcare', 'access', 'and', 'racial/ethnic', 'minorities', 'experience', 'workplace', 'discrimination']
['people', 'from', 'lower', 'socioeconomic', 'backgrounds', 'face', 'exclusion', 'in', 'settings', 'including', 'housing', 'health', 'care', 'and', 'education']


In [11]:
# Per ogni frase idonea si calcola la media della distribuzione di frequenza dei token:
# somma delle frequenze nel corpus dei token della frase, diviso il numero di token

def media_frequenze(frase, freq_corpus):
    somma_frequenze = sum(freq_corpus[t] for t in frase)
    return somma_frequenze / len(frase)

# Si associa a ogni frase la sua media
medie = [(frase, media_frequenze(frase, freq_corpus)) for frase in frasi_idonee]

# 5.1 - Frase con media più alta
frase_max = max(medie, key=lambda x: x[1])
print("=== Frase con media delle frequenze più ALTA ===")
print(f"Media: {frase_max[1]:.2f}")
print(f"Frase: {' '.join(frase_max[0])}")

# 5.2 - Frase con media più bassa
frase_min = min(medie, key=lambda x: x[1])
print("\n=== Frase con media delle frequenze più BASSA ===")
print(f"Media: {frase_min[1]:.2f}")
print(f"Frase: {' '.join(frase_min[0])}")

=== Frase con media delle frequenze più ALTA ===
Media: 259.50
Frase: the dependent variable is attitude towards the target

=== Frase con media delle frequenze più BASSA ===
Media: 12.60
Frase: s1 table shows sample demographics


In [12]:
# Modello di Markov di ordine 3: la probabilità di una frase è data dalla
# probabilità del primo trigramma, moltiplicata per le probabilità condizionate
# P(parola | 3 parole precedenti) di ogni token successivo
# P(w4 | w1 w2 w3) = freq(w1 w2 w3 w4) / freq(w1 w2 w3)

# Distribuzioni di frequenza dei 3-grammi e 4-grammi sull'intero corpus (token in minuscolo)
freq_trigrammi = FreqDist(ngrams(tokens_lower, 3))
freq_quadrigrammi = FreqDist(ngrams(tokens_lower, 4))
N = len(tokens_lower)

# Funzione che calcola la probabilità di una frase secondo il modello di ordine 3
def probabilita_markov3(frase):
    trigrammi_frase = list(ngrams(frase, 3))
    quadrigrammi_frase = list(ngrams(frase, 4))

    # Se la frase è troppo corta per avere almeno un trigramma, non è valutabile
    if len(trigrammi_frase) == 0:
        return 0.0

    # Inizializzazione con la probabilità del primo trigramma: freq(primo trigramma) / N
    prob = freq_trigrammi[trigrammi_frase[0]] / N

    # Moltiplicazione per le probabilità condizionate di ogni token successivo
    for q in quadrigrammi_frase:
        trigramma_precedente = q[:3]
        freq_tri = freq_trigrammi[trigramma_precedente]
        if freq_tri == 0:
            prob_cond = 0.0
        else:
            prob_cond = freq_quadrigrammi[q] / freq_tri
        prob = prob * prob_cond
    return prob

# Calcolo della probabilità per ogni frase idonea e selezione della più probabile
prob_frasi = [(frase, probabilita_markov3(frase)) for frase in frasi_idonee]
frase_piu_probabile = max(prob_frasi, key=lambda x: x[1])

print("=== Frase con probabilità più alta secondo il modello di Markov di ordine 3 ===")
print(f"Probabilità: {frase_piu_probabile[1]}")
print(f"Frase: {' '.join(frase_piu_probabile[0])}")

=== Frase con probabilità più alta secondo il modello di Markov di ordine 3 ===
Probabilità: 4.924410301866352e-05
Frase: similarly racism harms people s health and healthcare access and racial/ethnic minorities experience workplace discrimination


## 6. Percentuale di stopwords

In [13]:
# Percentuale di stopwords nel corpus rispetto al totale dei token
# La lista di stopwords è quella fornita da NLTK per l'inglese

stopwords_eng = set(stopwords.words("english"))

# Conteggio dei token che sono stopwords (confronto in minuscolo)
num_stopwords = sum(1 for t in tokens_lower if t in stopwords_eng)

percentuale_stopwords = num_stopwords / len(tokens_lower) * 100

print(f"Numero di token: {len(tokens_lower)}")
print(f"Numero di stopwords: {num_stopwords}")
print(f"Percentuale di stopwords: {percentuale_stopwords:.2f}%")

Numero di token: 20307
Numero di stopwords: 6903
Percentuale di stopwords: 33.99%


## 7. Frequenza dei pronomi personali

In [14]:
# Frequenza d'uso dei pronomi personali nel corpus
# I pronomi personali sono identificati dal tag PoS "PRP"

# 7.1 - Numero di pronomi personali sul totale dei token
pronomi_personali = [token for token, tag in tokens_pos if tag == "PRP"]
num_pronomi = len(pronomi_personali)
percentuale_pronomi = num_pronomi / len(tokens_pos) * 100

# 7.2 - Numero medio di pronomi personali per frase
media_pronomi_frase = num_pronomi / len(frasi)

print(f"Numero di pronomi personali: {num_pronomi}")
print(f"Percentuale sul totale dei token: {percentuale_pronomi:.2f}%")
print(f"Numero medio di pronomi personali per frase: {media_pronomi_frase:.2f}")

Numero di pronomi personali: 197
Percentuale sul totale dei token: 0.97%
Numero medio di pronomi personali per frase: 0.24


## 8. Named Entity Recognition

In [15]:
# Estrazione delle Named Entity (NE) del corpus con ne_chunk
# ne_chunk genera un albero in cui le entità nominate sono raggruppate in nodi
# che hanno un attributo 'label' contenente il tipo di entità

albero_NE = ne_chunk(tokens_pos)

# Le entità vengono raccolte in un dizionario: la chiave è il tipo di entità,
# il valore è la lista delle entità di quel tipo trovate nel corpus
entita_per_classe = {}
for nodo in albero_NE:
    if hasattr(nodo, 'label'):
        tipo = nodo.label()
        entita = " ".join(token for token, tag in nodo.leaves())
        if tipo not in entita_per_classe:
            entita_per_classe[tipo] = []
        entita_per_classe[tipo].append(entita)

# Per ciascuna classe si stampano le 20 entità più frequenti, ordinate per frequenza decrescente
for tipo, lista_entita in entita_per_classe.items():
    print(f"\n=== {tipo} (top-20) ===")
    for entita, freq in FreqDist(lista_entita).most_common(20):
        print(f"{entita}: {freq}")


=== ORGANIZATION (top-20) ===
MHPs: 37
N400: 36
Black: 35
White: 14
EEG: 14
ERP: 11
P300: 10
Nature: 9
Participants: 8
Black Americans: 7
US: 6
ERPs: 6
OSF: 5
African American: 5
ANOVA: 5
Blatant: 5
Congruent: 5
People: 5
Social: 4
GSNI: 4

=== PERSON (top-20) ===
Black: 25
Alaska Native: 4
Likert: 4
Brusa: 4
Armenia Brazil: 3
See Fig: 3
Future: 3
Jin: 3
Brazil: 2
Values Survey: 2
Cronbach: 2
Study: 2
Allport: 2
Duffy: 2
Furthermore: 2
People: 2
Nature: 2
Mustafa: 2
Class: 1
Status: 1

=== GPE (top-20) ===
Black: 17
Russia: 10
Italian: 10
Germany: 9
India: 9
Australia: 6
Caucasian: 4
Russian: 3
Studies: 3
Asian: 3
American: 3
Armenia: 2
Brazil: 2
Uygur: 2
Netherlands: 2
N400: 2
Incongruent: 2
Neutral: 2
Unipark: 1
Turpanjian: 1

=== GSP (top-20) ===
US: 2
Russia: 2

=== FACILITY (top-20) ===
White: 46


## 9. LMI delle Named Entity multi-token

In [16]:
# Per le Named Entity composte da almeno due token si calcola la LMI tra i loro token
# LMI(w1...wn) = freq(w1...wn) * log( p(w1...wn) / (p(w1)*p(w2)*...*p(wn)) )
# dove le probabilità sono stimate come frequenza nel corpus diviso il numero totale di token

# Frequenza dei singoli token e dimensione del corpus (token in minuscolo)
freq_token = FreqDist(tokens_lower)
N = len(tokens_lower)

# Raccolta delle Named Entity composte da almeno due token
# Ogni entità è rappresentata come lista dei suoi token (in minuscolo)
ne_multitoken = []
for nodo in albero_NE:
    if hasattr(nodo, 'label'):
        token_entita = [token.lower() for token, tag in nodo.leaves()]
        if len(token_entita) >= 2:
            ne_multitoken.append(token_entita)

# Frequenza di ogni entità multi-token, calcolata sul numero di volte in cui
# la sequenza è stata riconosciuta come Named Entity nel corpus
freq_ne = FreqDist(tuple(ne) for ne in ne_multitoken)

# Calcolo della LMI per ogni entità multi-token distinta
risultati_lmi = {}
for entita in set(tuple(ne) for ne in ne_multitoken):
    freq_congiunta = freq_ne[entita]
    p_congiunta = freq_congiunta / N

    # Prodotto delle probabilità dei singoli token
    p_singoli = 1.0
    for token in entita:
        p_singoli = p_singoli * (freq_token[token] / N)

    lmi_valore = freq_congiunta * math.log2(p_congiunta / p_singoli)
    risultati_lmi[entita] = lmi_valore

# Stampa delle entità ordinate per LMI decrescente
print("=== LMI delle Named Entity composte da almeno 2 token ===")
for entita, valore in sorted(risultati_lmi.items(), key=lambda x: x[1], reverse=True):
    print(f"{' '.join(entita)}: {valore:.4f}")

=== LMI delle Named Entity composte da almeno 2 token ===
eastern europeans arabs africans asians south americans roma: 89.9390
african american: 52.3085
alaska native: 49.2388
arabs africans asians south americans: 48.0099
chicago face database: 46.2551
ruhr university bochum germany data: 45.0871
jacobs university bremen germany data: 45.0871
american indian: 42.8989
black americans: 40.3126
ant neuro: 38.1742
black americans consequently black americans: 31.9376
human development: 31.7182
armenia australia brazil india: 29.2288
black americans introduction stereotyping: 28.9860
ready set go: 27.0344
internal review boards: 27.0344
armenia brazil: 26.9274
economics moscow russia: 24.9189
see fig: 22.5070
university ethics committee: 22.2975
lightspeed australia brazil: 21.8380
quick discrimination index: 21.0344
asa4 software results: 20.4901
gender social norms index: 20.2350
erp difference wave: 20.1116
afz fz: 19.6357
policy analysis armenia: 19.4051
international income distribut